In [81]:
import argparse
import json
import re
import sys
from biom import load_table

control_pattern_default = r"(blank|control|neg|ntc|water|reagent)"
table_path = "../data/V4_stool/V4_stool.biom"
ambiguities_path = "../data/V4_stool/V4_stool.biom.ambiguities"

# Compile control regex
is_control = re.compile(control_pattern_default, re.I).search

# Load table
try:
    table = load_table(table_path)
except Exception as e:
    sys.exit(f"[error] failed to load BIOM table '{table_path}': {e}")

# Load ambiguities mapping
try:
    with open(ambiguities_path) as f:
        ambig = json.load(f)
except Exception as e:
    sys.exit(f"[error] failed to read ambiguities JSON '{ambiguities_path}': {e}")

# Current sample IDs present in table
present = set(table.ids(axis="sample"))

keep = set()
sample_to_idx = {sample_id: i for i, sample_id in enumerate(table.ids(axis="sample"))}
idx_to_sample = {i: sample_id for sample_id, i in sample_to_idx.items()}
sample_depths = table.sum(axis='sample')

# Helper: total observation count (sequencing depth) for a sample
def depth(sample_id: str) -> float:
    # table.sum(axis='sample', ids=[id]) returns a 1-element ndarray
    return float(sample_depths[sample_to_idx[sample_id]])



In [76]:

print(len(present), "samples before filtering")

# for sample_id in present:
#     print(f"sample_id: {sample_id}")
#     print(depth(sample_id))
#     break

42500 samples before filtering


In [ ]:
# import random
# def test_depth(sample_id: str) -> float:
#     return random.randint(1000, 10000)

In [ ]:
# kept items from replicate sets
clip_count = 5000
kept_from_replicates = 0
replicate_total = len([run for runs in ambig.values() for run in runs])

for canon, runs in ambig.items():
    # print(f"canon: {canon}, runs: {runs}")
    
    if is_control(canon):
        print(f"  skipping control run: {canon}")
        continue
    avail = [run for run in runs if run in present]
    if not avail:
        print(f"  no available samples for run: {canon}")
        continue
    best = max(avail, key=depth)
    if depth(best) < clip_count:
        print(f"  skipping low-depth run: {best} (depth: {depth(best)})")
        continue
    # print(best)
    keep.add(best)
    kept_from_replicates += 1
print(f"kept {kept_from_replicates} samples out of {replicate_total} ambiguous samples.")
    




  skipping low-depth run: 10564.DC6MEYNB.134305 (depth: 3513.0)
  skipping low-depth run: 10564.O9G6DQH2.134305 (depth: 2745.0)
  skipping low-depth run: 10317.X00215889.147023 (depth: 28.0)
  skipping low-depth run: 10317.000001525.131452 (depth: 131.0)
  skipping low-depth run: 10317.X00214414.147023 (depth: 51.0)
  skipping low-depth run: 10564.QCMUUEDW.134305 (depth: 3417.0)
  skipping low-depth run: 2086.1325500812A.131614 (depth: 9.0)
  skipping low-depth run: 2086.1325904517.129439 (depth: 15.0)
  skipping low-depth run: 10317.000020791.131424 (depth: 85.0)
  skipping low-depth run: 10317.000038111.131418 (depth: 4315.0)
  skipping low-depth run: 10317.X00215244.156882 (depth: 80.0)
  skipping low-depth run: 2086.1325401895.129439 (depth: 4.0)
  skipping low-depth run: 10564.74ZDKQVK.134305 (depth: 4977.0)
  skipping low-depth run: 10564.YZL1C941.134305 (depth: 4883.0)
  skipping low-depth run: 10564.8HDXIBAS.134305 (depth: 3127.0)
  skipping low-depth run: 10564.CE3XUVX2.134305

In [83]:
# subtract all ambiguous sample from present, then add back kept ones
ambig_runs = {run for runs in ambig.values() for run in runs}
unambiguous = present - ambig_runs

print(f"{len(present)} total samples")
print(f"{len(ambig_runs)} ambiguous samples")
print(f"{len(unambiguous)} unambiguous samples")

for sample_id in unambiguous:
    if is_control(sample_id):
        print(f"skipping control run: {sample_id}")
        continue
    if depth(sample_id) < clip_count:
        print(f"  skipping low-depth run: {sample_id} (depth: {depth(sample_id)})")
        continue
    keep.add(sample_id)

print(f"{len(keep)} total samples after filtering.")

42500 total samples
6744 ambiguous samples
35756 unambiguous samples
  skipping low-depth run: 894.YY1830.lane2.NoIndex.L002.128416 (depth: 13.0)
  skipping low-depth run: 10317.000012201.128816 (depth: 2703.0)
  skipping low-depth run: 12906.Milk.T6.171.132803 (depth: 4816.0)
  skipping low-depth run: 1958.Antibioticday5.10.86.128677 (depth: 1003.0)
  skipping low-depth run: 10317.000053363.132132 (depth: 3069.0)
  skipping low-depth run: 10093.s3d42p6.127811 (depth: 1134.0)
  skipping low-depth run: 1958.Preantibiotic.4.62.128677 (depth: 1414.0)
  skipping low-depth run: 10317.000042933.132132 (depth: 3035.0)
  skipping low-depth run: 11113.1122.127587 (depth: 25.0)
  skipping low-depth run: 10317.000011077.128816 (depth: 3734.0)
  skipping low-depth run: 10317.000016182.128434 (depth: 2560.0)
  skipping low-depth run: 10317.000040346.128247 (depth: 1498.0)
  skipping low-depth run: 10317.000015195.128434 (depth: 4.0)
skipping control run: 11914.St.neg.GLD065.155591
  skipping low-de

In [84]:
from biom.util import biom_open
filter_fn = lambda val, id_, md: id_ in keep
new_table = table.filter(filter_fn, inplace=False)
out_path = "cleaned_table.biom"
with biom_open(out_path, "w") as f:
    new_table.to_hdf5(f, "cleaned table")
print(f"Wrote cleaned BIOM table to: {out_path}")

Wrote cleaned BIOM table to: cleaned_table.biom


In [ ]:
# import numpy as np
# from biom.table import Table
# data = np.asarray([[0, 0, 1], [1, 3, 42]])
# table = Table(data, ['O1', 'O2'], ['S1', 'S2', 'S3'],
#               [{'full_genome_available': True},
#                {'full_genome_available': False}],
#               [{'sample_type': 'a'}, {'sample_type': 'a'},
#                {'sample_type': 'b'}])
# print(table)
# kept = set()
# kept.add('S1')
# kept.add('S2')
# filter_fn = lambda val, id_, md: id_ in kept
# new_table = table.filter(filter_fn, inplace=False)
# print(new_table)

# Constructed from biom file
#OTU ID	S1	S2	S3
O1	0.0	0.0	1.0
O2	1.0	3.0	42.0
# Constructed from biom file
#OTU ID	S1	S2
O1	0.0	0.0
O2	1.0	3.0
